In [1]:
!pip install elasticsearch

In [6]:
# docker run -d \
#  --name elasticsearch \
#  --platform linux/arm64 \
#  -p 9200:9200 \
#  -e "discovery.type=single-node" \
#  -e "xpack.security.enabled=false" \
#  docker.elastic.co/elasticsearch/elasticsearch:9.2.4

from elasticsearch import Elasticsearch, helpers
import uuid
import time

INDEX_NAME = "biblical_women"

# 2. Setup Mapping (Ensures fields can be used for Aggregations)
def setup_index():
    if not client.indices.exists(index=INDEX_NAME):
        # We define "fields" as both text and keyword
        mapping = {
            "mappings": {
                "properties": {
                    "name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
                    "role": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
                    "lineage": {"type": "text", "fields": {"keyword": {"type": "keyword"}}}
                }
            }
        }
        client.indices.create(index=INDEX_NAME, body=mapping)
        print(f"Created index: {INDEX_NAME}")

# 3. Data Generator
def generate_women(count):
    famous_women = [
        {"name": "Sarah", "role": "Matriarch", "lineage": "Abraham"},
        {"name": "Deborah", "role": "Judge", "lineage": "Ephraim"},
        {"name": "Esther", "role": "Queen", "lineage": "Benjamin"},
        {"name": "Mary", "role": "Theotokos", "lineage": "David"},
        {"name": "Junia", "role": "Apostle", "lineage": "Early Church"}
    ]
    for i in range(count):
        source = famous_women[i] if i < len(famous_women) else {
            "name": f"Woman_Character_{i}",
            "role": "Historical Figure",
            "lineage": "Unknown"
        }
        yield {"_index": INDEX_NAME, "_id": str(uuid.uuid4()), "_source": source}

# 4. INQUIRY LOGIC
def run_inquiries():
    print("\n--- INQUIRY 1: Fuzzy Search ---")
    # Finds "Debora" even if misspelled
    resp = client.search(index=INDEX_NAME, query={"match": {"name": {"query": "Debora", "fuzziness": "AUTO"}}})
    for hit in resp['hits']['hits']:
        print(f"Found: {hit['_source']['name']} ({hit['_source']['role']})")

    print("\n--- INQUIRY 2: Role Distribution (Aggregation) ---")
    # Summarizes the 100,000 records
    agg_query = {
        "size": 0,
        "aggs": {
            "roles": {"terms": {"field": "role.keyword"}}
        }
    }
    resp = client.search(index=INDEX_NAME, body=agg_query)
    for bucket in resp['aggregations']['roles']['buckets']:
        print(f"Role: {bucket['key']} | Count: {bucket['doc_count']}")

# --- EXECUTION ---
setup_index()

print("Ingesting 100,000 items (Small Batch processing)...")
helpers.bulk(client, generate_women(100000), chunk_size=1000)
print("Ingestion complete.")

# Wait a moment for Elastic to refresh the index
time.sleep(2) 
run_inquiries()

Ingesting 100,000 items (Small Batch processing)...
Ingestion complete.

--- INQUIRY 1: Fuzzy Search ---
Found: Deborah (Judge)
Found: Deborah (Judge)
Found: Deborah (Judge)
Found: Deborah (Judge)

--- INQUIRY 2: Role Distribution (Aggregation) ---
Role: Historical Figure | Count: 399980
Role: Apostle | Count: 4
Role: Judge | Count: 4
Role: Matriarch | Count: 4
Role: Queen | Count: 4
Role: Theotokos | Count: 4
